# Module 20 — Real-Time Feature Engineering: Rolling Statistics

> **Track 2 · Revolut Fintech Specialization**  
> **Difficulty**: Intermediate  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Pandas, NumPy, Scikit-learn, Plotly  
> **Prerequisite**: Module 2 (Advanced GroupBy), Module 11 (Rule-Based Fraud)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Streaming Data](#3-synthetic-streaming-data)
4. [Rolling Window Features](#4-rolling-window-features)
5. [Anomaly Detection with Rolling Features](#5-anomaly-detection-with-rolling-features)
6. [Feature Importance Analysis](#6-feature-importance-analysis)
7. [Visualization](#7-visualization)
8. [Incremental Update Patterns](#8-incremental-update-patterns)
9. [Key Business Takeaway](#9-key-business-takeaway)

---
## §1 · Business Context

### Why rolling features matter for fintech?

Real-time feature engineering enables **instant fraud detection**:
- **Rolling statistics**: Moving averages, standard deviations over time windows.
- **Adaptive thresholds**: Personalized limits based on user behavior.
- **Streaming updates**: Incremental computation for low-latency inference.

**Applications at Revolut**:
1. **Fraud detection**: Flag transactions 3x above user's rolling average.
2. **Spending insights**: Track 7-day vs 30-day spending trends.
3. **Credit scoring**: Rolling income/expense patterns.

**Technical challenges**:
- **Latency**: Features must update in < 10ms for real-time scoring.
- **Memory**: Store rolling windows efficiently per user.
- **Consistency**: Handle out-of-order events and late arrivals.

In this notebook we will:
1. Generate synthetic streaming transaction data.
2. Compute rolling features (7-day, 30-day windows).
3. Detect anomalies using rolling statistics.
4. Demonstrate incremental update patterns for production.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import time
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.ensemble import IsolationForest

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Streaming Data

In [ ]:
# Generate synthetic streaming transaction data
N_TRANSACTIONS = 500000
N_USERS = 10000

# Create transaction stream
timestamps = pd.date_range('2024-01-01', periods=N_TRANSACTIONS, freq='1s')
user_ids = np.random.randint(0, N_USERS, N_TRANSACTIONS)
amounts = np.random.exponential(50, N_TRANSACTIONS)
categories = np.random.choice(['grocery', 'restaurant', 'transport', 'online', 'cash'], N_TRANSACTIONS)

df_stream = pd.DataFrame({
    'timestamp': timestamps,
    'user_id': user_ids,
    'amount': amounts,
    'category': categories
})

# Sort by timestamp
df_stream = df_stream.sort_values('timestamp').reset_index(drop=True)

print(f"✓ Generated streaming data:")
print(f"  Transactions: {len(df_stream):,}")
print(f"  Users: {N_USERS:,}")
print(f"  Time range: {df_stream['timestamp'].min()} to {df_stream['timestamp'].max()}")

---
## §4 · Rolling Window Features

In [ ]:
# Calculate rolling features for each user
print("Computing rolling features (this may take moment)...")

# Group by user and compute rolling stats
def compute_rolling_features(group):
    group = group.sort_values('timestamp')
    
    # 7-day rolling statistics
    group['rolling_7d_mean'] = group['amount'].rolling(window='7D', on='timestamp').mean()
    group['rolling_7d_std'] = group['amount'].rolling(window='7D', on='timestamp').std()
    group['rolling_7d_sum'] = group['amount'].rolling(window='7D', on='timestamp').sum()
    group['rolling_7d_count'] = group['amount'].rolling(window='7D', on='timestamp').count()
    
    # 30-day rolling statistics
    group['rolling_30d_mean'] = group['amount'].rolling(window='30D', on='timestamp').mean()
    group['rolling_30d_std'] = group['amount'].rolling(window='30D', on='timestamp').std()
    
    # Exponential moving average
    group['ema_7d'] = group['amount'].ewm(span=7).mean()
    
    return group

# Apply to a sample (full dataset would be too slow for demo)
sample_users = np.random.choice(N_USERS, 1000, replace=False)
df_sample = df_stream[df_stream['user_id'].isin(sample_users)]
df_features = df_sample.groupby('user_id', group_keys=False).apply(compute_rolling_features)

print(f"✓ Computed rolling features for {len(sample_users):,} users")
print(f"  Features: 7-day MA, STD, SUM, COUNT, 30-day MA, STD, EMA")

---
## §5 · Anomaly Detection with Rolling Features

In [ ]:
# Detect anomalies using rolling features
def detect_anomalies(row):
    """Flag transactions that deviate from rolling patterns."""
    if pd.isna(row['rolling_7d_mean']) or pd.isna(row['rolling_7d_std']):
        return 0
    
    # Z-score based anomaly detection
    z_score = abs(row['amount'] - row['rolling_7d_mean']) / (row['rolling_7d_std'] + 1e-6)
    return 1 if z_score > 3 else 0

df_features['is_anomaly'] = df_features.apply(detect_anomalies, axis=1)

anomaly_rate = df_features['is_anomaly'].mean()
print(f"✓ Anomaly detection complete")
print(f"  Anomaly rate: {anomaly_rate*100:.2f}%")
print(f"  Anomalies detected: {df_features['is_anomaly'].sum():,}")

---
## §6 · Feature Importance Analysis

In [ ]:
# Analyze which rolling features are most predictive
# Prepare features
rolling_features = ['rolling_7d_mean', 'rolling_7d_std', 'rolling_7d_sum',
                    'rolling_30d_mean', 'rolling_30d_std', 'ema_7d']

X = df_features[rolling_features].dropna()
iso_forest = IsolationForest(contamination=0.05, random_state=42)
predictions = iso_forest.fit_predict(X)

# Count anomalies
n_anomalies = (predictions == -1).sum()
print(f"✓ Isolation Forest anomalies: {n_anomalies:,} ({n_anomalies/len(X)*100:.2f}%)")

---
## §7 · Visualization

In [ ]:
# Visualize rolling features and anomalies
fig = make_subplots(rows=2, cols=1, subplot_titles=['Transaction Amount with Rolling Mean', 'Anomaly Detection'])

# Sample one user
user_sample = df_features[df_features['user_id'] == sample_users[0]].head(100)

fig.add_trace(go.Scatter(
    x=user_sample['timestamp'],
    y=user_sample['amount'],
    mode='lines+markers',
    name='Amount',
    opacity=0.6
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=user_sample['timestamp'],
    y=user_sample['rolling_7d_mean'],
    mode='lines',
    name='7-day Rolling Mean',
    line=dict(color='red', width=2)
), row=1, col=1)

# Anomalies
anomalies = user_sample[user_sample['is_anomaly'] == 1]
fig.add_trace(go.Scatter(
    x=anomalies['timestamp'],
    y=anomalies['amount'],
    mode='markers',
    name='Anomalies',
    marker=dict(color='red', size=10, symbol='x')
), row=2, col=1)

fig.update_layout(height=600, showlegend=True)
fig.show()

print("💡 Rolling features enable:")
print("   - Real-time anomaly detection")
print("   - Personalized thresholds per user")
print("   - Adaptive to changing spending patterns")

---
## §8 · Incremental Update Patterns

In [ ]:
# Demonstrate incremental feature updates
print("=" * 70)
print("INCREMENTAL FEATURE UPDATE PATTERN")
print("=" * 70)

print("""
For production systems, use incremental updates:

class RollingFeatureStore:
    def __init__(self, window_size=7):
        self.window_size = window_size
        self.user_stats = {}
    
    def update(self, user_id, amount, timestamp):
        if user_id not in self.user_stats:
            self.user_stats[user_id] = {
                'amounts': [],
                'timestamps': []
            }
        
        stats = self.user_stats[user_id]
        stats['amounts'].append(amount)
        stats['timestamps'].append(timestamp)
        
        # Remove old entries outside window
        cutoff = timestamp - timedelta(days=self.window_size)
        while stats['timestamps'][0] < cutoff:
            stats['amounts'].pop(0)
            stats['timestamps'].pop(0)
        
        # Compute rolling stats
        return {
            'rolling_mean': np.mean(stats['amounts']),
            'rolling_std': np.std(stats['amounts']),
            'rolling_count': len(stats['amounts'])
        }
""")

---
## §9 · Key Business Takeaway

> ### 🎯 Action Items for Real-Time Feature Engineering
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Real-time fraud detection, spending insights, credit scoring |
> | **Features** | Rolling statistics (mean, std, sum, count) over time windows |
> | **Model** | Z-score or Isolation Forest for anomaly detection |
> | **Evaluation** | Anomaly rate, detection latency |
> | **Deployment** | Incremental updates with sliding windows |
>
> ### 💡 Production Implementation
>
> 1. **Use efficient data structures**:
>    ```
>    deque for sliding windows (O(1) append/remove)
>    Redis for distributed feature store
>    ```
>
> 2. **Window sizes for different use cases**:
>    | Use Case | Window Size | Update Frequency |
>    |----------|-------------|------------------|
>    | Fraud detection | 7 days | Per transaction |
>    | Spending insights | 30 days | Daily |
>    | Credit scoring | 90 days | Monthly |
>
> 3. **Handle edge cases**:
>    - New users: Use population averages as fallback
>    - Sparse activity: Use longer windows or decay factors
>    - Out-of-order events: Buffer and reorder
>
> ### 🔑 Key Takeaways
>
> 1. **Rolling features enable personalized thresholds** for each user.
> 2. **7-day and 30-day windows** capture different behavioral patterns.
> 3. **Incremental updates** are essential for real-time systems.
> 4. **Z-score anomalies** are simple but effective for fraud detection.
> 5. **Combine multiple window sizes** for robust feature engineering.